In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# Linear Regression
### Statistical Fitting for Biochemical Data — Protein Assay & Calibration Curves

**Author:** Omar Hamouda

**Environment:** Python 3.x | Jupyter Notebook | Anaconda

---
## 📌 Overview
This notebook introduces **linear regression** using `scipy.stats`.
We will fit a **calibration curve** from protein assay data,
extract the best-fit statistics (slope, intercept, R², p-value),
and use the equation to calculate **unknown protein concentrations**
from absorbance measurements.

## 📚 Table of Contents
1. Importing Libraries (os, numpy, pandas, scipy.stats)
2. Loading Data with Pandas
3. Linear Regression with `scipy.stats.linregress`
4. Calculating Protein Concentrations in Unknown Samples
5. Adding Results to the DataFrame
6. Filtering Out-of-Range Values
7. Exercise: Groundwater Survey (pH vs. Bicarbonate)

In [ ]:
# Import the libraries we need
import os                          # File paths
import numpy as np                 # Numerical operations
import pandas as pd                # Tabular data
from scipy import stats            # Statistics & linear regression

### Loading the Protein Assay Data

The file `data/protein_assay.csv` contains **standard curve** data
from a protein assay: known protein concentrations and their
corresponding absorbance readings.

We use the same pattern from Lesson 05:
- `os.path.join()` → build the path
- `os.path.exists()` → verify it exists
- `pd.read_csv()` → load it into a DataFrame

In [ ]:
# Build the path to the protein assay file
protein_file = os.path.join('data', 'protein_assay.csv')
print(protein_file)

# Verify it exists (best practice)
if os.path.exists(protein_file):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the protein assay data into a DataFrame
results_df = pd.read_csv(protein_file)

# Display the DataFrame
results_df

### Extracting X and Y Data

For linear regression, we need two separate Series:
- **xdata** (independent variable) → `Protein Concentration (mg/mL)`
- **ydata** (dependent variable) → `A595` (absorbance)

We access each column using `df['column_name']` — the pattern from
Lesson 05.

In [ ]:
# Extract the independent (X) and dependent (Y) variables
xdata = results_df['Protein Concentration (mg/mL)']   # X: known concentrations
ydata = results_df['A595']                            # Y: absorbance readings

# Verify both Series
print("X data (concentration):")
print(xdata)
print("\nY data (absorbance):")
print(ydata)

### Linear Regression with `scipy.stats.linregress`

`scipy.stats.linregress()` performs a **least-squares linear regression**
and returns 5 values:
- **slope** — the gradient (sensitivity of the assay)
- **intercept** — the y-intercept
- **r_value** — correlation coefficient
- **p_value** — statistical significance
- **std_err** — standard error of the slope

In biochemistry, this is used for **calibration curves** — for example,
converting absorbance readings into protein concentrations.

In [ ]:
# Perform linear regression
slope, intercept, r_value, p_value, std_err = stats.linregress(xdata, ydata)

# Display the results with clear labels
print(f"Slope = {slope}")
print(f"Intercept = {intercept}")
print(f"R-squared = {r_value**2}")
print(f"P value = {p_value}")
print(f"Standard error = {std_err}")

### Calculating Unknown Protein Concentrations

Now we use the **calibration curve equation** to calculate the
concentration of unknown protein samples from their absorbance.

Rearranging `A = slope × C + intercept`:

The sample data is stored in `data/protein_samples.csv`.


In [ ]:
# Build the path to the samples file
samples_file = os.path.join('data', 'protein_samples.csv')
print(samples_file)

# Verify it exists
if os.path.exists(samples_file):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the sample data into a DataFrame
samples_df = pd.read_csv(samples_file)

# Display the DataFrame
samples_df

In [ ]:
# Apply the calibration equation to calculate protein concentration
protein_conc = (samples_df['A-595'] - intercept) / slope

# Display the calculated concentrations
print(protein_conc)

### Adding the Concentrations as a New Column

Adding a column to a DataFrame is simple: `df['new_column'] = values`.
Here, we add `ProtConc` — the calculated protein concentration for
each sample (in mg/mL).

In [ ]:
# Add the calculated concentrations as a new column
samples_df['ProtConc'] = protein_conc

# Display the updated DataFrame
samples_df

### Eliminating Out-of-Range Values

The unknown samples have absorbances that must fall **within** the
range covered by the standards. If an unknown sample's absorbance
is below the lowest standard or above the highest standard, its
calculated concentration is **not experimentally valid**.

We flag such samples with `np.nan` (Not a Number) — a special float
value that represents a missing value. This preserves the numeric
data type of the column.

In [ ]:
# Mark samples with absorbances OUTSIDE the standards' range as NaN
samples_df.loc[samples_df['A-595'] < 0.285, 'ProtConc'] = np.nan
samples_df.loc[samples_df['A-595'] > 1.118, 'ProtConc'] = np.nan

# Display the updated DataFrame
samples_df

### Best Practice: Dynamic Range Detection

Instead of hard-coding the min/max values, we can compute them
**directly from the standard curve data**. This makes the code
**robust** — if the standards change, the code adapts automatically.

We use:
- `.min()` → lowest absorbance among standards
- `.max()` → highest absorbance among standards

In [ ]:
print("results_df columns:", results_df.columns.tolist())
print("samples_df columns:", samples_df.columns.tolist())

In [ ]:
# Get the min and max absorbance from the STANDARDS (results_df)
# NOTE: results_df uses 'A595' (no hyphen), samples_df uses 'A-595' (with hyphen)
min_abs = results_df['A595'].min()
max_abs = results_df['A595'].max()

print(f"Standards range: {min_abs} – {max_abs}")

# Mark samples outside the standards' range as NaN
samples_df.loc[samples_df['A-595'] < min_abs, 'ProtConc'] = np.nan
samples_df.loc[samples_df['A-595'] > max_abs, 'ProtConc'] = np.nan

# Display the final DataFrame
samples_df

---
## 🧪 Exercise: Ground Water Survey

A survey of well water in Texas recorded **bicarbonate levels (ppm)**
and **pH** for multiple wells. We will use pandas + SciPy to perform
linear regression and assess the relationship.

### 🎯 Goal
- X (independent): `Bicarbonate` (ppm in well water)
- Y (dependent): `pH`

### 📌 Expected Output

In [ ]:
# Build the path to the groundwater file
groundwater_file = os.path.join('data', 'Ground_water.csv')
print(groundwater_file)

# Verify it exists
if os.path.exists(groundwater_file):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the groundwater data into a DataFrame
groundwater_df = pd.read_csv(groundwater_file)

# Show the column names first (lesson learned from KeyError!)
print("Columns:", groundwater_df.columns.tolist())

# Display the first 5 rows
groundwater_df.head()

In [ ]:
# Extract X (Bicarb) and Y (pH)
xdata = groundwater_df['Bicarb']
ydata = groundwater_df['pH']

# Verify
print("X data (Bicarb):")
print(xdata.head())
print("\nY data (pH):")
print(ydata.head())

In [ ]:
# Perform linear regression on groundwater data
slope_gw, intercept_gw, r_value_gw, p_value_gw, std_err_gw = stats.linregress(xdata, ydata)

# Display the results
print(f"Slope = {slope_gw}")
print(f"Intercept = {intercept_gw}")
print(f"R-squared = {r_value_gw**2}")
print(f"P value = {p_value_gw}")
print(f"Standard error = {std_err_gw}")

---
## 📝 Summary & Key Takeaways

In this notebook, we performed **linear regression** on real
biochemical data — building a **calibration curve** for protein
quantification and analyzing a **groundwater survey** dataset.

### What we covered:
- **Imports** — `numpy`, `pandas`, and `scipy.stats`.
- **Loading Data** — `pd.read_csv()` with `os.path.join()`.
- **Extracting Series** — `df['column']` to get X and Y.
- **Linear Regression** — `scipy.stats.linregress(x, y)` returns 5 values.
- **Applying Results** — used slope/intercept to calculate unknowns.
- **Adding Columns** — `df['new_col'] = values`.
- **Filtering Outliers** — with `.loc[condition, column] = np.nan`.
- **Dynamic Range Detection** — `.min()` and `.max()` instead of hardcoded values.

### Results produced:

| Dataset | R² | P value | Interpretation |
|---------|-----|---------|----------------|
| **Protein Assay** | 0.9947 | 1.06e-05 | Excellent linear fit |
| **Groundwater (pH vs Bicarb)** | 0.1153 | 0.0495 | Weak but significant |

**Calibration equation for protein assay:**  A595 = 0.8454 × Concentration + 0.1279


---

## ⚠️ Important Notes

### On Linear Regression
- `linregress()` returns **5 values**: slope, intercept, r_value, p_value, std_err.
- **R²** measures the fraction of variance explained (closer to 1 = better).
- **P value < 0.05** indicates statistical significance.
- A **negative slope** is possible — it just means Y decreases with X.

### On Column Names
- **Always check `df.columns` after `read_csv()`** — column names can
  differ between files (e.g., `A595` vs `A-595`).
- This single step prevents `KeyError`.

### On Filtering
- **Hardcoded values are brittle** — they break when standards change.
- **Prefer `.min()` / `.max()`** for dynamic range detection.
- **Use `np.nan`** (not strings) to mark invalid values — it preserves dtype.

### On Real Data
- Not every relationship is strong — **R² = 0.11 is real science**.
- A significant P value doesn't mean the relationship is **useful**.
- Always **think critically** about your results.

---

## 🔜 Next Steps
- **Visualize** the calibration curve and groundwater data (Matplotlib).
- **Nonlinear regression** — fit the Michaelis-Menten equation directly.
- **Multi-variable regression** — explore relationships among many variables.
- **Statistical tests** — t-tests, ANOVA, confidence intervals.
- **Cross-validation** — verify your calibration on independent data.

---

## 🧬 Biological Context

### Protein Assay
The Bradford/BCA/Lowry assays are **colorimetric methods** used in
every biochemistry lab to quantify protein concentration. The
**Beer-Lambert Law** (A = ε·c·l) is the physical basis — absorbance
is proportional to concentration. A calibration curve converts
absorbance into concentration for unknown samples.

### Groundwater Survey
pH and bicarbonate (HCO₃⁻) are key indicators of **water alkalinity**
and **buffering capacity**. The weak correlation observed here
(R² = 0.11) reflects the complex chemistry of natural waters —
other ions (Ca²⁺, Mg²⁺, CO₃²⁻) also influence pH.

**Lesson:** Real-world data is rarely as clean as textbook examples.